# CH101 free hybrid quality strategies

This notebook runs each new strategy at most once: TRELLIS is guarded by a GPU/VRAM/CUDA/license preflight, while the reference-fitted semantic proxy can run with CPU Blender. Both candidates use the same refine, evaluate, score, strict visual QA, and ranking path. A failed strategy is never silently retried, and all Unity/Production gates remain locked.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

CHARACTER_CODE = 'CH101'
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = os.environ.get('RE_CAMP_BLENDER_TOOLS_REF', 'feature/ch101-free-ai3d-autobuild')
TOOLS_COMMIT_EXPECTED = os.environ.get('RE_CAMP_BLENDER_TOOLS_COMMIT', '')
ART_REPO_URL = 'https://github.com/siri2677/re-camp.git'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
TRELLIS_REPO_URL = 'https://github.com/microsoft/TRELLIS.git'
TRELLIS_COMMIT = '442aa1e1afb9014e80681d3bf604e8d728a86ee7'
TRELLIS_STRATEGY_ID = 'TRELLIS_SINGLE_VIEW_V001'
SEMANTIC_STRATEGY_ID = 'SEMANTIC_PROXY_REFERENCE_FITTED_V001'
RUNTIME_NAME = os.environ.get('RE_CAMP_RUNTIME', '').strip().lower()
if not RUNTIME_NAME:
    RUNTIME_NAME = 'kaggle' if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/working').is_dir() else 'colab'
CONTENT_ROOT = Path('/kaggle/working' if RUNTIME_NAME == 'kaggle' else '/content')
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
OUTPUT_ROOT = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE / 'hybrid'
REFERENCE_DIR = OUTPUT_ROOT / 'reference-views'
EVALUATION_DIR = OUTPUT_ROOT / 'evaluation'
SEMANTIC_DIR = OUTPUT_ROOT / 'semantic-proxy'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidates'
REVIEW_DIR = OUTPUT_ROOT / 'review'
GATES = {'sourceStatus': 'AI_GENERATED_CANDIDATE_NOT_PRODUCTION', 'gateB': 'PENDING_HUMAN_REVIEW', 'unityInputAllowed': False, 'productionPromotionAllowed': False}
print({'runtime': RUNTIME_NAME, 'trellisStrategy': TRELLIS_STRATEGY_ID, 'semanticStrategy': SEMANTIC_STRATEGY_ID, **GATES})

In [ ]:
def run(command, *, cwd=None, check=True, env=None):
    command = [str(part) for part in command]
    print('RUN:', ' '.join(command))
    result = subprocess.run(command, cwd=cwd, env=env, check=False)
    if check and result.returncode:
        raise RuntimeError(f'command failed ({result.returncode}): {command}')
    return result

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not (TOOLS_DIR / '.git').is_dir():
    run(['git', 'clone', '--branch', TOOLS_REF, TOOLS_REPO_URL, TOOLS_DIR])
run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
TOOLS_COMMIT = subprocess.check_output(['git', '-C', TOOLS_DIR, 'rev-parse', 'HEAD'], text=True).strip()
if TOOLS_COMMIT_EXPECTED: assert TOOLS_COMMIT == TOOLS_COMMIT_EXPECTED
if not (ART_DIR / '.git').is_dir():
    run(['git', 'clone', ART_REPO_URL, ART_DIR])
run(['git', '-C', ART_DIR, 'fetch', 'origin', ART_COMMIT])
run(['git', '-C', ART_DIR, 'checkout', '--detach', ART_COMMIT])
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py', '--art-root', ART_DIR, '--output-dir', REFERENCE_DIR, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
REFERENCE_MANIFEST = REFERENCE_DIR / 'reference-views-manifest.json'
SEMANTIC_HANDOFF = OUTPUT_ROOT / 'semantic-reconstruction-inputs.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_semantic_reconstruction_handoff.py', '--art-root', ART_DIR, '--output', SEMANTIC_HANDOFF, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json', '--character', CHARACTER_CODE])
print({'referenceManifest': str(REFERENCE_MANIFEST), 'semanticHandoff': str(SEMANTIC_HANDOFF), **GATES})

In [ ]:
# CPU Blender is only for the semantic proxy; it is never installed for a blocked TRELLIS path.
BLENDER_BIN = shutil.which('blender')
if not BLENDER_BIN and os.environ.get('RE_CAMP_INSTALL_CPU_BLENDER', '1') == '1':
    install = run(['apt-get', 'update', '-qq'], check=False)
    if install.returncode == 0:
        run(['apt-get', 'install', '-y', '-qq', 'blender', 'xvfb'], check=False)
    BLENDER_BIN = shutil.which('blender')
BLENDER_LAUNCHER = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
print({'blender': BLENDER_BIN or 'BLOCKED_BLENDER_AUTHORING_ENVIRONMENT', 'cpuSemanticProxyAllowed': bool(BLENDER_BIN), **GATES})

In [ ]:
# quality_progress_gate runs before any candidate execution. A rejected strategy cannot be repeated.
# BLOCKED_PROVIDER_PREFLIGHT and REGENERATE_REQUIRED are terminal recorded states for this run.
ORCHESTRATION_REPORT = OUTPUT_ROOT / 'hybrid-quality-orchestration.json'
orchestration_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'hybrid_quality_orchestrator.py', '--art-root', ART_DIR, '--output', ORCHESTRATION_REPORT, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json', '--character', CHARACTER_CODE, '--score-dir', EVALUATION_DIR, '--history-record', TOOLS_DIR / 'docs' / 'records' / 'ch101-ai3d' / '2026-08-28-wonder3d-selection-root-cause-v074.json']
run(orchestration_command)
orchestration = json.loads(ORCHESTRATION_REPORT.read_text(encoding='utf-8'))
assert orchestration['unityInputAllowed'] is False and orchestration['productionPromotionAllowed'] is False
TRELLIS_PLAN = orchestration['strategies'][TRELLIS_STRATEGY_ID]
SEMANTIC_PLAN = orchestration['strategies'][SEMANTIC_STRATEGY_ID]
TRELLIS_RUN_ALLOWED = TRELLIS_PLAN['runAllowed'] is True
SEMANTIC_RUN_ALLOWED = SEMANTIC_PLAN['runAllowed'] is True
(OUTPUT_ROOT / 'trellis-preflight.json').write_text(json.dumps(TRELLIS_PLAN['preflight'], ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print({'trellisStatus': TRELLIS_PLAN['status'], 'semanticStatus': SEMANTIC_PLAN['status'], 'selectedStrategies': orchestration['selectedStrategies'], **GATES})

In [ ]:
# Run the CPU-capable semantic proxy only when its one-shot quality gate is ready.
CANDIDATE_MANIFESTS = []
if SEMANTIC_RUN_ALLOWED:
    if not BLENDER_BIN:
        raise RuntimeError('BLOCKED_BLENDER_AUTHORING_ENVIRONMENT')
    run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'build_ch101_semantic_proxy.py', '--', '--output-dir', SEMANTIC_DIR, '--reference-report', SEMANTIC_HANDOFF, '--art-root', ART_DIR, '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json', '--render'])
    SEMANTIC_REPORT = SEMANTIC_DIR / 'semantic-proxy-report.json'
    semantic_report = json.loads(SEMANTIC_REPORT.read_text(encoding='utf-8'))
    assert semantic_report['unityInputAllowed'] is False and semantic_report['productionPromotionAllowed'] is False
    semantic_mesh = semantic_report.get('mesh') or semantic_report['glb']
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', semantic_mesh, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', CANDIDATE_DIR, '--provider', 'semanticProxy', '--strategy-id', SEMANTIC_STRATEGY_ID, '--source-stage', 'SEMANTIC_PROXY_REFERENCE_FITTED', '--candidate-label', '001', '--metadata-json', SEMANTIC_REPORT, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
    CANDIDATE_MANIFESTS.append(CANDIDATE_DIR / 'candidate-manifest.json')
else:
    print({'semanticProxy': 'SKIPPED', 'reason': SEMANTIC_PLAN['status'], **GATES})
print({'candidateManifests': [str(path) for path in CANDIDATE_MANIFESTS], **GATES})

In [ ]:
# TRELLIS runs at most once and is registered into the same candidate list.
TRELLIS_REPORT = OUTPUT_ROOT / 'trellis-one-shot-run-report.json'
if TRELLIS_RUN_ALLOWED:
    TRELLIS_DIR = CONTENT_ROOT / 'provider-TRELLIS'
    if not (TRELLIS_DIR / '.git').is_dir():
        run(['git', 'clone', TRELLIS_REPO_URL, TRELLIS_DIR])
    run(['git', '-C', TRELLIS_DIR, 'fetch', '--depth', '1', 'origin', TRELLIS_COMMIT])
    run(['git', '-C', TRELLIS_DIR, 'checkout', '--detach', TRELLIS_COMMIT])
    trellis_command = os.environ.get('RE_CAMP_TRELLIS_COMMAND', '').split()
    if trellis_command:
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_trellis_candidate.py', '--provider-repo', TRELLIS_DIR, '--input-image', REFERENCE_DIR / 'CH101_front.png', '--output-dir', OUTPUT_ROOT / 'trellis-output', '--preflight', OUTPUT_ROOT / 'trellis-preflight.json', '--output-report', TRELLIS_REPORT, '--execute', '--'] + trellis_command, check=False)
    else:
        TRELLIS_REPORT.write_text(json.dumps({'status': 'BLOCKED_PROVIDER_ENTRYPOINT_UNVERIFIED', 'strategyId': TRELLIS_STRATEGY_ID, **GATES}, indent=2) + '\n', encoding='utf-8')
else:
    TRELLIS_REPORT.write_text(json.dumps({'status': TRELLIS_PLAN['status'], 'strategyId': TRELLIS_STRATEGY_ID, 'preflight': TRELLIS_PLAN['preflight'], **GATES}, indent=2) + '\n', encoding='utf-8')
trellis_result = json.loads(TRELLIS_REPORT.read_text(encoding='utf-8'))
if trellis_result.get('status') == 'TRELLIS_EXECUTED':
    for index, mesh_path in enumerate(trellis_result.get('meshOutputs', []), start=1):
        trellis_candidate_dir = CANDIDATE_DIR / 'trellis' / f'{index:03d}'
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', mesh_path, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', trellis_candidate_dir, '--provider', 'trellis', '--strategy-id', TRELLIS_STRATEGY_ID, '--source-stage', 'TRELLIS_SINGLE_VIEW_RESEARCH', '--candidate-label', f'{index:03d}', '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
        CANDIDATE_MANIFESTS.append(trellis_candidate_dir / 'candidate-manifest.json')
print({'trellisStatus': trellis_result.get('status'), 'trellisCandidatesRegistered': len([path for path in CANDIDATE_MANIFESTS if 'trellis' in str(path).lower()]), **GATES})
# Every registered candidate uses the existing refine -> evaluate -> score -> strict visual QA path.
SCORE_REPORTS = []
for manifest_path in CANDIDATE_MANIFESTS:
    payload = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
    entry = payload['candidates'][0]
    candidate_id = entry['candidateId']
    candidate_output = EVALUATION_DIR / candidate_id
    candidate_output.mkdir(parents=True, exist_ok=True)
    refined_glb = candidate_output / f'{candidate_id}_refined.glb'
    refined_blend = candidate_output / f'{candidate_id}_refined_NOT_PRODUCTION.blend'
    refinement_report = candidate_output / 'refinement-report.json'
    candidate_provider = entry.get('provider', 'semanticProxy')
    candidate_strategy = entry.get('strategyId', SEMANTIC_STRATEGY_ID)
    run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py', '--', '--candidate', entry['modelPath'], '--output-glb', refined_glb, '--output-blend', refined_blend, '--report', refinement_report, '--provider', candidate_provider, '--attempt', '1', '--parent-sha256', entry['sha256'], '--material-mode', 'preserve'])
    refinement_payload = json.loads(refinement_report.read_text(encoding='utf-8'))
    evaluation_candidate = Path(refinement_payload.get('refinedTransportPath') or entry['modelPath'])
    evaluation_report = candidate_output / 'evaluation-report.json'
    normalized_blend = candidate_output / f'{candidate_id}_normalized_NOT_PRODUCTION.blend'
    run(BLENDER_LAUNCHER + [BLENDER_BIN, '-b', '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py', '--', '--candidate', evaluation_candidate, '--candidate-id', candidate_id, '--strategy-id', candidate_strategy, '--output-dir', candidate_output, '--report', evaluation_report, '--normalized-blend', normalized_blend, '--integrity-blend', refined_blend])
    score_report = candidate_output / 'candidate-score.json'
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py', '--reference-manifest', REFERENCE_MANIFEST, '--evaluation-report', evaluation_report, '--candidate-manifest', manifest_path, '--output', score_report, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE])
    SCORE_REPORTS.append(score_report)
if SCORE_REPORTS:
    REVIEW_DIR.mkdir(parents=True, exist_ok=True)
    ASSISTED_REVIEW = REVIEW_DIR / 'assisted-visual-review.json'
    review_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'build_assisted_visual_review.py', '--output', ASSISTED_REVIEW, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE]
    for report_path in SCORE_REPORTS:
        review_command.extend(['--score-report', report_path])
    run(review_command)
    RANKING_MANIFEST = OUTPUT_ROOT / 'ranking-manifest.json'
    ranking_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'rank_candidates.py', '--output', RANKING_MANIFEST, '--contract', TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json', '--character', CHARACTER_CODE, '--assisted-visual-review', ASSISTED_REVIEW]
    for report_path in SCORE_REPORTS:
        ranking_command.extend(['--score-report', report_path])
    run(ranking_command)
    ranking = json.loads(RANKING_MANIFEST.read_text(encoding='utf-8'))
    assert ranking['unityInputAllowed'] is False and ranking['productionPromotionAllowed'] is False
    print({'strictVisualQA': ranking.get('status'), 'selectedCandidate': ranking.get('selectedCandidate'), **GATES})
else:
    ranking = {'status': 'NO_CANDIDATE_STRATEGY_READY', **GATES}
    print(ranking)

In [ ]:
# The TRELLIS one-shot result and any registered meshes were handled before scoring.
print({'trellisResult': json.loads(TRELLIS_REPORT.read_text(encoding='utf-8')), **GATES})

In [ ]:
execution_report = {
    'schemaVersion': 'ch101-hybrid-quality-execution-v001',
    'character': CHARACTER_CODE,
    'toolsCommit': TOOLS_COMMIT,
    'artCommit': ART_COMMIT,
    'trellisProviderCommit': TRELLIS_COMMIT,
    'trellis': TRELLIS_PLAN,
    'semanticProxy': SEMANTIC_PLAN,
    'candidateManifests': [str(path) for path in CANDIDATE_MANIFESTS],
    'scoreReports': [str(path) for path in SCORE_REPORTS],
    'rankingManifest': str(OUTPUT_ROOT / 'ranking-manifest.json') if SCORE_REPORTS else None,
    'status': ranking.get('status', 'NO_CANDIDATE_STRATEGY_READY'),
    **GATES,
}
EXECUTION_REPORT = OUTPUT_ROOT / 'hybrid-quality-execution-report.json'
EXECUTION_REPORT.write_text(json.dumps(execution_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
archive = Path(shutil.make_archive(str(CONTENT_ROOT / f're-camp-{CHARACTER_CODE}-hybrid-NOT-PRODUCTION'), 'zip', OUTPUT_ROOT))
print({'executionReport': str(EXECUTION_REPORT), 'archive': str(archive), **GATES})
try:
    from google.colab import files
    files.download(str(archive))
except Exception:
    print('Browser download unavailable; preserve the archive before the session ends.')